In [2]:
# Build 1D-CNN IDS model
# 1D-CNN = trees 80 features as sequence
# Learns local patterns in network traffic

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import(
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report
)
import matplotlib.pyplot as plt
import time

print(f"TensorFlow version: {tf.__version__}")
print("Libraries loaded!")

2026-05-15 10:28:42.331309: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.16.2
Libraries loaded!


In [6]:
# Step 2: Load preprocessed data
print("Loading data...")

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/CICIDS-2017/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test =pd.read_csv(save_path + "y_test.csv").squeeze()

label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

print(f"Classes: {label_classes}")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print("Data loaded!")

Loading data...
Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Heartbleed', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack']
X_train: (1979513, 80)
X_test: (848363, 80)
Data loaded!


In [7]:
# Step 3: Encode labels
# CNN needs numbers not text labels
# BENIGN = 0, Bot = 1, DDOS 2 ect.
from sklearn.preprocessing import LabelEncoder

print("Encoding labels...")
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

num_classes = len(le.classes_)
print(f"Classes: {list(le.classes_)}")
print(f"Number of classes: {num_classes}")
print("Labels encoded!")


Encoding labels...
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Number of classes: 13
Labels encoded!


In [8]:
# Step 4: Reshape data for 1D_CNN
# CNN needs 3D input: (samples, steps, features)
# Current shape: (1979513,80)
# Need shape:    (1979513, 80, 1)

X_train_cnn = X_train.reshape(
    X_train.shape[0], X_train.shape[1],1)
X_test_cnn = X_test.reshape(
    X_test.shape[0], X_test.shape[1],1)

print(f"X_train, reshaped: {X_train_cnn.shape}")
print(f"X_test reshaped: {X_test_cnn.shape}")
print("Reshape complete!")


X_train, reshaped: (1979513, 80, 1)
X_test reshaped: (848363, 80, 1)
Reshape complete!


In [9]:
# Step 5: Build 1D-CNN model
# Conv1D = learn local feature patterns
# MaxPooling = reduces dimensions 
# Dense = final classification layers 

print("Building 1D-CNN model...")

model = keras.Sequential([
    # input layer
    keras.layers.Input(shape=(80,1)),

    # Covn layer 1: learn local patterns
    keras.layers.Conv1D(
        filters = 64,
        kernel_size = 3,
        activation = 'relu',
        padding = 'same'
    ),
    keras.layers.MaxPooling1D(pool_size=2),
    # Conv layer 2: learn deeper patterns 
    keras.layers.Conv1D(
        filters=123,
        kernel_size=3,
        activation='relu',
        padding='same'
    ),
    keras.layers.MaxPooling1D(pool_size=2),

    # Flatten and classify
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Building 1D-CNN model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 80, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 40, 123)        │        23,739 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 20, 123)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2460)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       315,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 13)             │         1,677 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 340,680 (1.30 MB)

 Trainable params: 340,680 (1.30 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Step 6: Train 1D-CNN model

print("Training 1D-CNN...")
print("This will take 30-60 mitunes...")

start_time = time.time()

history = model.fit(
    X_train_cnn, y_train_enc,
    epochs=5,
    batch_size=1024,
    validation_split=0.1,
    verbose=1
)

end_time = time.time()
cnn_train_time = round(end_time - start_time, 2)

print(f"Training complete!")
print(f"Training time: {cnn_train_time} second")


Training 1D-CNN...
This will take 30-60 mitunes...
Epoch 1/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 720s 411ms/step - accuracy: 0.9438 - loss: 0.1924 - val_accuracy: 0.9776 - val_loss: 0.0504
Epoch 2/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 628s 361ms/step - accuracy: 0.9772 - loss: 0.0528 - val_accuracy: 0.9802 - val_loss: 0.0446
Epoch 3/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 728s 418ms/step - accuracy: 0.9795 - loss: 0.0467 - val_accuracy: 0.9829 - val_loss: 0.0399
Epoch 4/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 500s 287ms/step - accuracy: 0.9813 - loss: 0.0424 - val_accuracy: 0.9848 - val_loss: 0.0368
Epoch 5/5
1740/1740 ━━━━━━━━━━━━━━━━━━━━ 586s 337ms/step - accuracy: 0.9832 - loss: 0.0389 - val_accuracy: 0.9850 - val_loss: 0.0360
Training complete!
Training time: 3166.32 second


In [14]:
# Step 7: Evaluate CNN model
print("Evaluating CNN model...")

start_time = time.time()
y_pred_enc = model.predict(
    X_test_cnn, batch_size=1024, verbose=1)
y_pred = np.argmax(y_pred_enc, axis=1)
end_time = time.time()

cnn_pred_time = round(end_time - start_time,2)

# Convert back to labels
y_pred_labels = le.inverse_transform(y_pred)

# Calcucate metrics
acc = accuracy_score(y_test, y_pred_labels)
p   = precision_score(y_test, y_pred_labels,
                     average='weighted')
r   = recall_score(y_test, y_pred_labels,
                  average='weighted')
f1  = f1_score(y_test, y_pred_labels,
              average='weighted')
print(f"\n=== 1D-CNN Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1_score:  {f1:.3f} ({f1*100:.2f}%)")
print(f"Prediction time: {cnn_pred_time}s")
print(f"\nDetailed Report:")
print(classification_report(y_test_enc, y_pred, target_names=label_classes))


Evaluating CNN model...
829/829 ━━━━━━━━━━━━━━━━━━━━ 70s 85ms/step

=== 1D-CNN Results ===
Accuracy:  0.9856 (98.56%)
Precision: 0.9866 (98.66%)
Recall:    0.9856 (98.56%)
F1_score:  0.986 (98.55%)
Prediction time: 78.18s

Detailed Report:
                  precision    recall  f1-score   support

          BENIGN       1.00      0.99      0.99    681396
             Bot       0.95      0.65      0.77       587
            DDoS       1.00      0.99      0.99     38408
   DoS GoldenEye       0.99      0.97      0.98      3088
        DoS Hulk       0.97      0.99      0.98     69037
DoS Slowhttptest       0.91      0.99      0.95      1650
   DoS slowloris       0.99      0.99      0.99      1739
     FTP-Patator       0.98      0.98      0.98      2380
      Heartbleed       1.00      1.00      1.00         3
    Infiltration       0.60      0.27      0.38        11
        PortScan       0.87      0.98      0.92     47641
     SSH-Patator       0.98      0.98      0.98      1769
     

In [15]:
# save label classes — for use in new dataset CNN notebooks
import json

model.save(save_path + "cnn_model.keras")

cnn_results = {
    "model": "1D-CNN",
    "accuracy": acc,
    "precision": p,
    "recall": r,
    "f1_score": f1,
    "training_time": 2872.22,
    "prediction_time": 69.89,
    "epochs": 5,
    "batch_size": 1024
}

with open(save_path + "cnn_results.json", "w") as f:
    json.dump(cnn_results, f, indent=4)

print("CNN model saved!")
print("Results saved!")

CNN model saved!
Results saved!
